### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

BLOG : 
https://www.kaggle.com/code/marcinrutecki/rag-mmr-search-in-langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [2]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()

# split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Step 2: FAISS Vector Store with Openai Embeddings
embedding_model = OpenAIEmbeddings()

# create FAISS vector store
vectorstore = FAISS.from_documents(chunks, embedding_model)

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Step 3: Create MMR Retirever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

mmr_retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000185F7F4ACF0>, search_type='mmr', search_kwargs={'k': 3})

In [5]:
from langchain_community.retrievers import BM25Retriever

# Step 4: Sparse Retriever(BM25)
sparse_retriever = BM25Retriever.from_documents(chunks)

sparse_retriever.k = 3 #top- k documents to retriever

sparse_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000185F7F4AF90>, k=3)

In [6]:
from langchain.retrievers import EnsembleRetriever

# step 5: combine dense and sparse retriever using EnsembleRetriever = hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[mmr_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000185F7F4ACF0>, search_type='mmr', search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000185F7F4AF90>, k=3)], weights=[0.7, 0.3])

In [7]:
from langchain.prompts import PromptTemplate

# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template(
"""
Answer the question based on the context provided.
Context: {context}
Question: {input}

"""
)

In [8]:
# LLM
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000185F81A0AD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000185F81A0EC0>, root_client=<openai.OpenAI object at 0x00000185F8C4F250>, root_async_client=<openai.AsyncOpenAI object at 0x00000185F84A0550>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [9]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(
    llm=llm, 
    prompt=prompt
)

# rag chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever, 
    combine_docs_chain=document_chain
)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000185F7F4ACF0>, search_type='mmr', search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000185F7F4AF90>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context provided.\nContext: {context}\nQuestion: {input}\n\n')
            

In [10]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)


print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain supports agents by allowing them to use tools like calculators, search APIs, or custom functions based on the instructions they receive. Agents in LangChain can also interact with external APIs and databases to enhance their capabilities. Additionally, LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.

In terms of memory, LangChain helps models retain previous interactions through its memory feature, making multi-turn conversations more coherent. It supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory. This allows agents to remember past conversations and information, improving the overall performance of the system.


In [11]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='2ca563e8-c310-4b18-b37c-18c28e3e8271', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='BM25 and vector-based retrieval can be combined in LangChain to support hybrid retrieval strategies.\nFAISS is a high-performance library for similarity search that LangChain leverages for efficient retrieval in RAG pipelines.'),
  Document(id='83ab4eb4-53da-407d-a67c-9897a176ef30', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='9355e45b-175d-4eca-91a2-9939017c7a10', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLan

## Quick Rule of Thumb:

1. Use Dense+Sparse → when keywords matter along with meaning.

2. Use Re-ranking → when you want the absolute best top-k docs (quality > speed).

3. Use MMR → when you want diversity in results (avoid repetition).